# Colab Pipeline: VI/EN Query -> Retrieval -> Qwen 8B Local Answer

Notebook này dùng để chạy trên Google Colab mà không cần import trực tiếp code backend local.

Luồng xử lý:

1. Cài thư viện cần thiết trên Colab.
2. Upload `chunks_standard_rag.jsonl` và file query nếu có.
3. Tải trực tiếp embedding model từ HuggingFace.
4. Encode chunks và build FAISS index trong runtime Colab.
5. Detect ngôn ngữ query, tùy chọn dịch query tiếng Việt sang tiếng Anh.
6. Retrieval bằng dense, BM25 hoặc hybrid.
7. Tải `Qwen/Qwen3-8B` về Colab và sinh câu trả lời local.
8. Xuất JSON và tải file kết quả về máy.


In [ ]:
# =========================
# 1. Cài thư viện trên Colab
# =========================

!pip -q install openai faiss-cpu sentence-transformers transformers accelerate bitsandbytes tqdm

import json
import math
import os
import re
import time
from collections import Counter, defaultdict
from datetime import datetime
from getpass import getpass
from pathlib import Path
from typing import Any

# Dự án dùng PyTorch cho sentence-transformers, không cần TensorFlow.
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

import faiss
import numpy as np
import torch
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.auto import tqdm

print("Đã cài và import thư viện xong.")

In [ ]:
# =========================
# 2. Cấu hình Colab pipeline
# =========================

# File chunks bắt buộc. Upload từ repo local: data/chunks/chunks_standard_rag.jsonl
CHUNKS_PATH = Path("/content/chunks_standard_rag.jsonl")

# File query tùy chọn. Upload từ repo local: data/evaluation/traveler_need_queries_500.jsonl
QUERY_PATH = Path("/content/traveler_need_queries_500.jsonl")

# File output cuối cùng.
OUTPUT_PATH = Path("/content/colab_vi_to_en_retrieval_qwen8b_answers.json")

# Embedding model sẽ được tải trực tiếp từ HuggingFace về Colab.
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Chọn retriever: dense, bm25 hoặc hybrid.
RETRIEVER_NAME = "hybrid"
TOP_K = 5
CANDIDATE_K = 20

# Số query cần chạy. Tăng lên 100/500 khi API ổn định.
LIMIT = 20

# Mặc định tắt dịch VI -> EN để giảm API call. Embedding model hiện tại là multilingual.
TRANSLATE_VI_QUERY = False

# Backend sinh câu trả lời: local, api hoặc extractive.
# local: tải open-source LLM từ HuggingFace về Colab và chạy trực tiếp.
# api: gọi OpenAI-compatible API như OpenRouter/OpenAI/GitHub.
# extractive: không dùng LLM, chỉ tóm tắt trực tiếp từ retrieved chunks.
GENERATION_BACKEND = "local"

# Local LLM mặc định theo yêu cầu: Qwen 8B.
# Khuyến nghị chạy Colab GPU. Nếu thiếu VRAM, đổi xuống Qwen/Qwen2.5-3B-Instruct hoặc Qwen/Qwen2.5-1.5B-Instruct.
LOCAL_LLM_MODEL_NAME = "Qwen/Qwen3-8B"
LOCAL_LLM_LOAD_IN_4BIT = True
LOCAL_LLM_ENABLE_THINKING = False
LOCAL_LLM_MAX_NEW_TOKENS = 900

# Model chat API. Chỉ dùng khi GENERATION_BACKEND = "api".
CHAT_MODEL = "openai/gpt-4o-mini"

# Endpoint provider: openrouter, openai hoặc github.
API_PROVIDER = "openrouter"
API_BASE_URL = ""  # Để trống để dùng default theo provider.

# Nếu để rỗng, notebook sẽ đọc Colab Secrets hoặc hỏi nhập key bằng getpass.
API_KEYS_TEXT = """"""

print("Cấu hình xong.")
print("CHUNKS_PATH:", CHUNKS_PATH)
print("QUERY_PATH:", QUERY_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("EMBEDDING_MODEL_NAME:", EMBEDDING_MODEL_NAME)
print("RETRIEVER:", RETRIEVER_NAME, "| TOP_K:", TOP_K, "| LIMIT:", LIMIT)


In [ ]:
# =========================
# 3. Upload dữ liệu và khởi tạo API client
# =========================

def upload_if_missing(path: Path, description: str) -> None:
    """Upload file lên Colab nếu file chưa tồn tại."""

    if path.exists():
        print(f"Đã có {description}: {path}")
        return

    from google.colab import files

    print(f"Chưa có {description}. Hãy upload file tương ứng.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError(f"Chưa upload {description}.")

    uploaded_name = next(iter(uploaded.keys()))
    uploaded_path = Path("/content") / uploaded_name
    if uploaded_path != path:
        uploaded_path.rename(path)
    print(f"Đã lưu {description}: {path}")


def split_keys(value: str | None) -> list[str]:
    """Tách danh sách API key phân cách bằng dấu phẩy hoặc xuống dòng."""

    if not value:
        return []
    return [item.strip() for item in re.split(r"[,\n]+", value) if item.strip()]


def read_colab_secret(name: str) -> str | None:
    """Đọc key từ Colab Secrets nếu có."""

    try:
        from google.colab import userdata

        value = userdata.get(name)
        return value if value else None
    except Exception:
        return None


def resolve_api_config(api_keys_text: str, provider: str, api_base_url: str) -> tuple[str, list[str]]:
    """Lấy endpoint và API keys cho OpenAI-compatible API."""

    inline_keys = split_keys(api_keys_text)
    secret_keys = (
        split_keys(read_colab_secret("OPENROUTER_API_KEYS"))
        or split_keys(read_colab_secret("OPENAI_API_KEYS"))
        or split_keys(read_colab_secret("OPENAI_API_KEY"))
        or split_keys(read_colab_secret("GITHUB_TOKEN"))
    )
    api_keys = inline_keys or secret_keys

    if not api_keys and GENERATION_BACKEND == "api":
        raw_key = getpass("Nhập API key để gọi chat model qua API, hoặc Enter nếu muốn bỏ qua API: ")
        api_keys = split_keys(raw_key)

    if not api_keys:
        return "", []

    first_key = api_keys[0]
    if first_key.startswith("sk-or-v1-"):
        provider = "openrouter"
    elif first_key.startswith("ghp_") or first_key.startswith("github_pat_"):
        provider = "github"

    if provider == "openrouter":
        return api_base_url or "https://openrouter.ai/api/v1", api_keys
    if provider == "openai":
        return api_base_url or "https://api.openai.com/v1", api_keys
    return api_base_url or "https://models.github.ai/inference", api_keys


class RotatingChatClient:
    """Client OpenAI-compatible có retry và đổi key khi rate limit."""

    def __init__(self, base_url: str, api_keys: list[str], model: str) -> None:
        self.base_url = base_url.rstrip("/")
        self.api_keys = api_keys
        self.model = model
        self.key_index = 0

    def _client(self) -> OpenAI:
        return OpenAI(base_url=self.base_url, api_key=self.api_keys[self.key_index])

    def chat(self, messages: list[dict[str, str]], temperature: float = 0.2, max_tokens: int = 1200) -> str:
        retry_terms = ["too many requests", "rate", "quota", "limit", "429", "insufficient"]
        last_error: Exception | None = None
        max_attempts = max(1, len(self.api_keys) * 3)
        for attempt in range(max_attempts):
            try:
                response = self._client().chat.completions.create(
                    model=self.model,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens,
                )
                return response.choices[0].message.content or ""
            except Exception as exc:
                last_error = exc
                message = str(exc).lower()
                can_retry = any(term in message for term in retry_terms)
                if can_retry and attempt < max_attempts - 1:
                    if len(self.api_keys) > 1:
                        self.key_index = (self.key_index + 1) % len(self.api_keys)
                    wait_seconds = min(20, 2 * (attempt + 1))
                    print(f"API bị rate limit/quota. Đợi {wait_seconds}s rồi thử lại...")
                    time.sleep(wait_seconds)
                    continue
                raise
        raise RuntimeError(f"Không gọi được chat model: {last_error}")


def load_local_llm(model_name: str):
    """Tải local LLM từ HuggingFace về Colab."""

    if GENERATION_BACKEND != "local":
        return None, None

    print("Đang tải local LLM:", model_name)
    print("GPU khả dụng:", torch.cuda.is_available())

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    quantization_config = None
    if torch.cuda.is_available() and LOCAL_LLM_LOAD_IN_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto" if torch.cuda.is_available() else None,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        quantization_config=quantization_config,
        trust_remote_code=True,
    )
    model.eval()
    print("Đã tải local LLM xong.")
    return tokenizer, model


def generate_local_chat(messages: list[dict[str, str]], max_new_tokens: int | None = None, temperature: float = 0.2) -> str:
    """Sinh text bằng local Qwen model."""

    if local_tokenizer is None or local_model is None:
        raise RuntimeError("Local LLM chưa được tải.")

    try:
        prompt = local_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=LOCAL_LLM_ENABLE_THINKING,
        )
    except TypeError:
        prompt = local_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = local_tokenizer(prompt, return_tensors="pt").to(local_model.device)
    generation_kwargs = {
        "max_new_tokens": max_new_tokens or LOCAL_LLM_MAX_NEW_TOKENS,
        "do_sample": temperature > 0,
        "temperature": temperature,
        "top_p": 0.9,
        "pad_token_id": local_tokenizer.eos_token_id,
    }
    with torch.no_grad():
        output_ids = local_model.generate(**inputs, **generation_kwargs)
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = local_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


upload_if_missing(CHUNKS_PATH, "chunks_standard_rag.jsonl")
if not QUERY_PATH.exists():
    print("Chưa có traveler_need_queries_500.jsonl. Notebook sẽ dùng query mẫu nếu bạn không upload ở cell sau.")

BASE_URL, API_KEYS = resolve_api_config(API_KEYS_TEXT, API_PROVIDER, API_BASE_URL)
chat_client = RotatingChatClient(BASE_URL, API_KEYS, CHAT_MODEL) if API_KEYS and GENERATION_BACKEND == "api" else None
local_tokenizer, local_model = load_local_llm(LOCAL_LLM_MODEL_NAME)
print("Generation backend:", GENERATION_BACKEND)
print("API endpoint:", BASE_URL or "Không dùng API")
print("Số API key đã nạp:", len(API_KEYS))

In [ ]:
# =========================
# 4. Tải embedding model và build retriever trên Colab
# =========================

TOKEN_PATTERN = re.compile(r"\w+", re.UNICODE)


def tokenize(text: str) -> list[str]:
    """Token hóa đơn giản cho BM25."""

    return [token.casefold() for token in TOKEN_PATTERN.findall(str(text or ""))]


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    """Đọc JSONL thành list dict."""

    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def normalize_text(text: Any) -> str:
    """Chuẩn hóa text ngắn trước khi đưa vào embedding/prompt."""

    return " ".join(str(text or "").split())


def build_retrieval_text(item: dict[str, Any]) -> str:
    """Lấy text tốt nhất cho retrieval từ chunk."""

    return normalize_text(item.get("retrieval_text") or item.get("source_text") or item.get("text") or item.get("content"))


class BM25Retriever:
    """BM25 thuần Python để chạy trên Colab không cần thêm dependency."""

    def __init__(self, items: list[dict[str, Any]], k1: float = 1.5, b: float = 0.75) -> None:
        self.items = items
        self.k1 = k1
        self.b = b
        self.doc_tokens = [tokenize(build_retrieval_text(item)) for item in items]
        self.doc_lengths = [len(tokens) for tokens in self.doc_tokens]
        self.avg_doc_length = sum(self.doc_lengths) / len(self.doc_lengths) if self.doc_lengths else 0.0
        self.term_frequencies = [Counter(tokens) for tokens in self.doc_tokens]
        self.document_frequencies: Counter[str] = Counter()
        for tokens in self.doc_tokens:
            self.document_frequencies.update(set(tokens))
        self.doc_count = len(items)

    def idf(self, term: str) -> float:
        df = self.document_frequencies.get(term, 0)
        return math.log(1 + (self.doc_count - df + 0.5) / (df + 0.5))

    def score_document(self, query_terms: list[str], index: int) -> float:
        score = 0.0
        frequencies = self.term_frequencies[index]
        doc_length = self.doc_lengths[index] or 1
        for term in query_terms:
            tf = frequencies.get(term, 0)
            if tf == 0:
                continue
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_length / max(self.avg_doc_length, 1e-9))
            score += self.idf(term) * numerator / denominator
        return score

    def search(self, query: str, top_k: int = 5) -> list[dict[str, Any]]:
        query_terms = tokenize(query)
        scored = [(index, self.score_document(query_terms, index)) for index in range(len(self.items))]
        scored = [(index, score) for index, score in scored if score > 0]
        scored.sort(key=lambda pair: pair[1], reverse=True)
        return [format_result(rank, score, self.items[index], "bm25") for rank, (index, score) in enumerate(scored[:top_k], 1)]


def format_result(rank: int, score: float, item: dict[str, Any], retriever_name: str) -> dict[str, Any]:
    """Chuẩn hóa output retrieval."""

    return {
        "rank": rank,
        "score": round(float(score), 6),
        "retriever": retriever_name,
        "chunk_id": item.get("chunk_id"),
        "document_id": item.get("document_id"),
        "chunk_index": item.get("chunk_index"),
        "document_title": item.get("document_title"),
        "source_url": item.get("source_url"),
        "language": item.get("language"),
        "source_text": item.get("source_text") or build_retrieval_text(item),
    }


chunks = load_jsonl(CHUNKS_PATH)
print("Số chunks:", len(chunks))

device = "cuda" if os.environ.get("COLAB_GPU") else "cpu"
print("Đang tải embedding model:", EMBEDDING_MODEL_NAME)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

texts = [build_retrieval_text(item) for item in chunks]
print("Đang encode chunks để tạo FAISS index...")
embeddings = embedding_model.encode(
    texts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.asarray(embeddings, dtype="float32")
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)
bm25_retriever = BM25Retriever(chunks)

print("Đã build FAISS index.")
print("Dimension:", embeddings.shape[1], "| Vectors:", faiss_index.ntotal)


def dense_search(query: str, top_k: int = 5, search_k: int | None = None) -> list[dict[str, Any]]:
    """Dense search bằng FAISS."""

    limit = search_k or top_k
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    query_embedding = np.asarray(query_embedding, dtype="float32")
    scores, ids = faiss_index.search(query_embedding, limit)
    results: list[dict[str, Any]] = []
    for score, item_id in zip(scores[0].tolist(), ids[0].tolist()):
        if item_id == -1:
            continue
        results.append(format_result(len(results) + 1, score, chunks[int(item_id)], "dense_faiss"))
        if len(results) >= top_k:
            break
    return results


def hybrid_search(query: str, top_k: int = 5, candidate_k: int = 20, rrf_k: int = 60) -> list[dict[str, Any]]:
    """Hybrid search bằng dense + BM25 với Reciprocal Rank Fusion."""

    dense_results = dense_search(query, top_k=candidate_k)
    bm25_results = bm25_retriever.search(query, top_k=candidate_k)
    by_chunk: dict[str, dict[str, Any]] = {}
    scores: Counter[str] = Counter()
    sources: dict[str, list[str]] = defaultdict(list)
    for source_name, result_set in [("dense", dense_results), ("bm25", bm25_results)]:
        for result in result_set:
            chunk_id = str(result.get("chunk_id"))
            by_chunk.setdefault(chunk_id, result)
            sources[chunk_id].append(source_name)
            scores[chunk_id] += 1.0 / (rrf_k + int(result["rank"]))
    ranked = sorted(scores.items(), key=lambda pair: pair[1], reverse=True)
    final_results: list[dict[str, Any]] = []
    for rank, (chunk_id, score) in enumerate(ranked[:top_k], 1):
        item = dict(by_chunk[chunk_id])
        item["rank"] = rank
        item["score"] = round(float(score), 6)
        item["retriever"] = "hybrid_bm25_dense_rrf"
        item["matched_sources"] = sources[chunk_id]
        final_results.append(item)
    return final_results


def search_chunks(query: str) -> list[dict[str, Any]]:
    """Search theo retriever đã chọn."""

    if RETRIEVER_NAME == "dense":
        return dense_search(query, top_k=TOP_K)
    if RETRIEVER_NAME == "bm25":
        return bm25_retriever.search(query, top_k=TOP_K)
    return hybrid_search(query, top_k=TOP_K, candidate_k=CANDIDATE_K)


In [ ]:
# =========================
# 5. Hàm xử lý query, prompt và answer
# =========================

VIETNAMESE_CHAR_PATTERN = re.compile(
    r"[ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ]",
    re.IGNORECASE,
)
VIETNAMESE_HINT_WORDS = {
    "ở", "đi", "đến", "nên", "gì", "món", "ăn", "chơi", "lịch", "trình",
    "khách", "sạn", "địa", "điểm", "tham", "quan", "bao", "nhiêu", "ngày",
    "buổi", "sáng", "tối", "mùa", "nào", "đẹp", "tiện", "gợi", "ý",
}
ENGLISH_HINT_WORDS = {
    "what", "where", "when", "which", "how", "best", "travel", "trip", "food",
    "restaurant", "hotel", "stay", "itinerary", "attraction", "visit", "things",
    "to", "do", "in", "near", "around", "recommend", "guide",
}


def detect_query_language(query: str) -> str:
    """Detect nhanh ngôn ngữ query."""

    text = query.strip().lower()
    if not text:
        return "unknown"
    if VIETNAMESE_CHAR_PATTERN.search(text):
        return "vi"
    tokens = re.findall(r"[a-zA-ZÀ-ỹ]+", text)
    if not tokens:
        return "unknown"
    vi_hits = sum(1 for token in tokens if token in VIETNAMESE_HINT_WORDS)
    en_hits = sum(1 for token in tokens if token in ENGLISH_HINT_WORDS)
    if vi_hits > en_hits:
        return "vi"
    if en_hits > 0:
        return "en"
    return "en"


def extract_json_object(text: str) -> dict[str, Any]:
    """Trích JSON object từ output LLM."""

    clean_text = text.strip()
    if clean_text.startswith("```"):
        clean_text = re.sub(r"^```(?:json)?", "", clean_text).strip()
        clean_text = re.sub(r"```$", "", clean_text).strip()
    try:
        return json.loads(clean_text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", clean_text, re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def generate_chat_response(messages: list[dict[str, str]], temperature: float = 0.2, max_tokens: int = 1200) -> str:
    """Sinh phản hồi bằng local Qwen hoặc API tùy cấu hình."""

    if GENERATION_BACKEND == "local":
        return generate_local_chat(messages, max_new_tokens=max_tokens, temperature=temperature)
    if GENERATION_BACKEND == "api" and chat_client:
        return chat_client.chat(messages, temperature=temperature, max_tokens=max_tokens)
    raise RuntimeError("Không có generation backend khả dụng.")


def translate_query_to_english(question_vi: str) -> dict[str, Any]:
    """Dịch query tiếng Việt sang query tiếng Anh nếu cần."""

    messages = [
        {
            "role": "system",
            "content": (
                "Bạn là bộ chuyển đổi truy vấn cho retrieval du lịch Việt Nam. "
                "Chuyển câu hỏi tiếng Việt thành query tiếng Anh ngắn, giàu keyword, giữ đúng địa danh và intent. "
                "Chỉ trả về JSON hợp lệ."
            ),
        },
        {
            "role": "user",
            "content": f'Câu hỏi tiếng Việt: {question_vi}\n\nSchema JSON: {{"english_query":"...","detected_locations":["..."],"intent":"..."}}',
        },
    ]
    raw = generate_chat_response(messages, temperature=0.0, max_tokens=300)
    parsed = extract_json_object(raw)
    parsed.setdefault("english_query", question_vi)
    parsed.setdefault("detected_locations", [])
    parsed.setdefault("intent", "general")
    return parsed


def prepare_retrieval_query(question: str) -> dict[str, Any]:
    """Chuẩn bị query retrieval: tiếng Anh dùng thẳng, tiếng Việt tùy chọn dịch."""

    detected_language = detect_query_language(question)
    if detected_language == "en" or not TRANSLATE_VI_QUERY:
        return {
            "retrieval_query": question,
            "retrieval_query_language": detected_language,
            "translation_status": "skipped",
            "translation_metadata": {"english_query": question, "reason": "Không dịch query trước retrieval."},
        }
    translation = translate_query_to_english(question)
    return {
        "retrieval_query": str(translation.get("english_query") or question),
        "retrieval_query_language": "en",
        "translation_status": "success",
        "translation_metadata": translation,
    }


def trim_text(text: str, max_chars: int) -> str:
    clean_text = " ".join(str(text or "").split())
    if len(clean_text) <= max_chars:
        return clean_text
    return clean_text[:max_chars].rstrip() + "..."


def build_context(chunks: list[dict[str, Any]], max_chunk_chars: int = 2200, max_context_chars: int = 11000) -> str:
    """Format retrieved chunks thành context cho model."""

    parts: list[str] = []
    current_length = 0
    for item in chunks:
        content = trim_text(str(item.get("source_text") or ""), max_chunk_chars)
        part = "\n".join(
            [
                f"[Nguồn {item.get('rank')} | score={item.get('score')} | retriever={item.get('retriever')} ]",
                f"Title: {item.get('document_title')}",
                f"URL: {item.get('source_url')}",
                f"Language: {item.get('language')}",
                "Content:",
                content,
            ]
        )
        if current_length + len(part) > max_context_chars:
            break
        parts.append(part)
        current_length += len(part)
    return "\n\n".join(parts)


def answer_with_context(question: str, retrieval_query: str, chunks: list[dict[str, Any]]) -> str:
    """Sinh câu trả lời tiếng Việt bằng local Qwen hoặc API dựa trên context."""

    context = build_context(chunks)
    messages = [
        {
            "role": "system",
            "content": (
                "Bạn là AI Assistant du lịch Việt Nam dùng Retrieval-Augmented Generation. "
                "Trả lời bằng tiếng Việt có dấu, chuyên nghiệp, rõ ràng và hữu ích. "
                "Chỉ dùng thông tin trong CONTEXT. Không bịa giá vé, giờ mở cửa, thời tiết hiện tại hoặc thông tin dễ thay đổi nếu context không nêu rõ."
            ),
        },
        {
            "role": "user",
            "content": (
                f"CONTEXT:\n{context}\n\n"
                f"CÂU HỎI NGƯỜI DÙNG:\n{question}\n\n"
                f"QUERY ĐÃ DÙNG ĐỂ RETRIEVE:\n{retrieval_query}\n\n"
                "YÊU CẦU TRẢ LỜI:\n"
                "- Trả lời tiếng Việt có dấu.\n"
                "- Viết như tư vấn viên du lịch chuyên nghiệp, đủ chi tiết, không trả lời cụt.\n"
                "- Cấu trúc gồm nhận định ngắn, gợi ý chính, lưu ý thực tế và nguồn tham khảo.\n"
                "- Nếu context không đủ, nói rõ phần nào thiếu.\n"
                "- Liệt kê tối đa 3 nguồn ở cuối nếu có URL."
            ),
        },
    ]
    return generate_chat_response(messages, temperature=0.25, max_tokens=LOCAL_LLM_MAX_NEW_TOKENS).strip()


def build_extractive_answer(question: str, retrieval_query: str, chunks: list[dict[str, Any]], error: Exception | None = None) -> str:
    """Fallback answer trực tiếp từ retrieved chunks khi không gọi được LLM."""

    lines = [
        "Dựa trên các nguồn liên quan nhất trong knowledge base, mình có thể tóm tắt nhanh như sau:",
        "",
        f"Nhu cầu: {question}",
        "",
        "Các thông tin nổi bật:",
    ]
    for index, item in enumerate(chunks[:4], 1):
        title = item.get("document_title") or "Không rõ tiêu đề"
        url = item.get("source_url") or ""
        preview = trim_text(str(item.get("source_text") or ""), 850)
        lines.append(f"{index}. {title}")
        if url:
            lines.append(f"   Nguồn: {url}")
        if preview:
            lines.append(f"   Nội dung liên quan: {preview}")
    lines.append("")
    lines.append(f"Query dùng để retrieval: {retrieval_query}")
    if error:
        lines.append("Ghi chú: LLM bị lỗi/rate limit nên đây là câu trả lời extractive từ context.")
    return "\n".join(lines)


def compact_sources(chunks: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Lưu metadata nguồn gọn trong output."""

    return [
        {
            "rank": item.get("rank"),
            "score": item.get("score"),
            "retriever": item.get("retriever"),
            "chunk_id": item.get("chunk_id"),
            "document_id": item.get("document_id"),
            "document_title": item.get("document_title"),
            "source_url": item.get("source_url"),
            "language": item.get("language"),
            "text_preview": trim_text(str(item.get("source_text") or ""), 400),
        }
        for item in chunks
    ]


In [ ]:
# =========================
# 6. Chạy pipeline và xuất file JSON
# =========================

def load_queries(path: Path, limit: int) -> list[dict[str, Any]]:
    """Đọc query file nếu có, nếu không dùng query mẫu."""

    if path.exists():
        rows = load_jsonl(path)
        return rows[:limit]

    sample_queries = [
        {"query_id": "sample_001", "query": "Ở Huế nên ăn gì và đi khu nào cho tiện?"},
        {"query_id": "sample_002", "query": "What should I do in Da Nang for two days?"},
        {"query_id": "sample_003", "query": "Hội An có trải nghiệm nào nên ưu tiên trong buổi sáng?"},
    ]
    return sample_queries[:limit]


queries = load_queries(QUERY_PATH, LIMIT)
results: list[dict[str, Any]] = []

for index, row in enumerate(queries, start=1):
    query_id = row.get("query_id") or f"query_{index:04d}"
    question = row.get("query") or row.get("question") or ""
    print(f"[{index}/{len(queries)}] {query_id}: {question}")

    step_errors: list[str] = []
    translation_metadata: dict[str, Any] = {}
    retrieval_query = question
    retrieval_query_language = detect_query_language(question)
    translation_status = "not_started"
    retrieved_chunks: list[dict[str, Any]] = []
    answer_vi = ""

    try:
        query_payload = prepare_retrieval_query(question)
        retrieval_query = str(query_payload["retrieval_query"])
        retrieval_query_language = str(query_payload["retrieval_query_language"])
        translation_status = str(query_payload["translation_status"])
        translation_metadata = dict(query_payload["translation_metadata"])
    except Exception as exc:
        step_errors.append(f"translation_error: {exc}")
        translation_status = "failed"
        translation_metadata = {"fallback_reason": "Dùng query gốc để retrieval vì dịch query lỗi."}

    try:
        retrieved_chunks = search_chunks(retrieval_query)
    except Exception as exc:
        step_errors.append(f"retrieval_error: {exc}")

    if retrieved_chunks:
        if GENERATION_BACKEND in {"local", "api"}:
            try:
                answer_vi = answer_with_context(question, retrieval_query, retrieved_chunks)
            except Exception as exc:
                step_errors.append(f"answer_error: {exc}")
                answer_vi = build_extractive_answer(question, retrieval_query, retrieved_chunks, exc)
        else:
            answer_vi = build_extractive_answer(question, retrieval_query, retrieved_chunks)
    else:
        step_errors.append("answer_skipped: Không có retrieved chunks.")

    status = "success" if answer_vi and not step_errors else "partial" if answer_vi else "error"
    results.append(
        {
            "query_id": query_id,
            "status": status,
            "error": " | ".join(step_errors) if step_errors else None,
            "question": question,
            "query_en": retrieval_query if retrieval_query_language == "en" else "",
            "retrieval_query": retrieval_query,
            "retrieval_query_language": retrieval_query_language,
            "translation_status": translation_status,
            "translation_metadata": translation_metadata,
            "answer_vi": answer_vi,
            "sources": compact_sources(retrieved_chunks),
            "input_metadata": {
                "user_intent": row.get("user_intent"),
                "category": row.get("category"),
                "locations": row.get("locations"),
                "traveler_profile": row.get("traveler_profile"),
                "time_context": row.get("time_context"),
            },
        }
    )
    time.sleep(0.5)

payload = {
    "metadata": {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "chunks_path": str(CHUNKS_PATH),
        "query_path": str(QUERY_PATH),
        "output_path": str(OUTPUT_PATH),
        "limit": LIMIT,
        "retriever": RETRIEVER_NAME,
        "top_k": TOP_K,
        "candidate_k": CANDIDATE_K,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "generation_backend": GENERATION_BACKEND,
        "local_llm_model": LOCAL_LLM_MODEL_NAME if GENERATION_BACKEND == "local" else None,
        "chat_model": CHAT_MODEL if chat_client else None,
        "api_base_url": BASE_URL if chat_client else None,
        "translate_vi_query": TRANSLATE_VI_QUERY,
        "generate_with_llm": GENERATION_BACKEND in {"local", "api"},
        "success_count": sum(1 for item in results if item["status"] == "success"),
        "partial_count": sum(1 for item in results if item["status"] == "partial"),
        "error_count": sum(1 for item in results if item["status"] == "error"),
    },
    "results": results,
}

OUTPUT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu output:", OUTPUT_PATH)
print("Success:", payload["metadata"]["success_count"], "| Partial:", payload["metadata"]["partial_count"], "| Error:", payload["metadata"]["error_count"])

# Tải file kết quả về máy local.
try:
    from google.colab import files

    files.download(str(OUTPUT_PATH))
except Exception as exc:
    print("Không tự download được file, hãy tải thủ công từ Files panel:", exc)

payload["results"][:1]
